In [10]:
import rasterio
import numpy as np
import torch
import joblib
from torch import nn

### Load the GeoTIFF with features

In [11]:
tif_path = 'data/GEE_exports/GEE_exports/features_2021_2023.tif'
with rasterio.open(tif_path) as src:
    # Read all bands (shape: bands, height, width)
    img = src.read()
    profile = src.profile
    transform = src.transform
    crs = src.crs
    height = img.shape[1]
    width = img.shape[2]
    n_bands = img.shape[0]

print(f"Image shape: {img.shape}")

Image shape: (5, 3203, 3256)


### Reshape image to a table of pixels (rows = pixels, cols = bands)

In [12]:
img_flat = img.reshape(n_bands, -1).T   
print(f"Number of pixels: {img_flat.shape[0]}")

Number of pixels: 10428968


### Load the scaler and the trained model

In [13]:
scaler = joblib.load('scaler.pkl')                 # fitted StandardScaler
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Define the model architecture (must match the one used during training)
class ForestLossNN(nn.Module):
    def __init__(self, input_dim, hidden_dims=[64, 32], dropout_rate=0.5):
        super(ForestLossNN, self).__init__()
        layers = []
        prev_dim = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(prev_dim, h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout_rate))
            prev_dim = h_dim
        layers.append(nn.Linear(prev_dim, 2))   # 2 classes: no loss / loss
        self.network = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.network(x)

# Instantiate model and load weights
input_dim = img_flat.shape[1]   # should be 5
model = ForestLossNN(input_dim).to(device)
model.load_state_dict(torch.load('forest_loss_nn.pth', map_location=device))
model.eval()
print("Model loaded successfully.")

Model loaded successfully.


### Scale the features using the pre‑fitted scaler

In [14]:
X = scaler.transform(img_flat)

### Run the model to get probabilities

In [15]:
X_tensor = torch.tensor(X, dtype=torch.float32).to(device)

# Run inference in batches to avoid memory issues (optional but safe)
batch_size = 10000
probabilities = []

with torch.no_grad():
    for i in range(0, len(X_tensor), batch_size):
        batch = X_tensor[i:i+batch_size]
        logits = model(batch)
        probs = torch.softmax(logits, dim=1)[:, 1]   # probability of class 1 (loss)
        probabilities.append(probs.cpu().numpy())

# Concatenate all batches
prob_flat = np.concatenate(probabilities, axis=0)
print(f"Probability array shape: {prob_flat.shape}")

Probability array shape: (10428968,)


### Reshape probabilities back to original image dimensions

In [16]:
prob_image = prob_flat.reshape(height, width)

### Handle no‑data pixels

In [17]:
nodata_mask = (img[0] == 0)
prob_image[nodata_mask] = np.nan

### Save the probability map as a new GeoTIFF

In [18]:
output_path = 'data/forest_loss_risk_2021_2023.tif'

# Update the profile for a single‑band float32 GeoTIFF
profile.update({
    'count': 1,
    'dtype': 'float32',
    'nodata': np.nan   # or a specific value like -9999
})

with rasterio.open(output_path, 'w', **profile) as dst:
    dst.write(prob_image.astype(np.float32), 1)

print(f"Risk map saved to: {output_path}")

Risk map saved to: data/forest_loss_risk_2021_2023.tif
